# Feature Engineering and ML Dataset Preparation
In this notebook, we execute Step 6 of the Real Estate ML pipeline: transforming raw/geocoded data into a clean, machine-learning-ready dataset. We select meaningful features, encode categorical variables, handle any residual outliers, and split our data to prevent data leakage.

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

## 1. Load and Inspect Data

In [ ]:
df = pd.read_csv('../data/processed/geocoded_property_data.csv')

print(f"Initial dataset shape: {df.shape}")
print(f"Columns:\n{list(df.columns)}")
print(f"\nMissing values:\n{df.isnull().sum()}")

display(df.head())

## 2 & 3. Identify Target and Select Features
We select our features based on EDA insights. We must explicitly DROP `price_per_sqft` to prevent **Target Leakage**, because the model would cheat by using a variable perfectly mathematically correlated to the target (Price).

In [ ]:
target = 'Price_INR'

# Removing price_per_sqft to avoid target leakage
features_removed = ['price_per_sqft']
df = df.drop(columns=features_removed)

print(f"Target variable: {target}")
print(f"Features removed: {features_removed} (Reason: Target leakage / calculated directly from price)")

# Ensure no lingering NaNs exist in coordinate columns
df = df.dropna()

# Define final features
categorical_features = ['Location', 'Property_Type']
numerical_features = ['Area_sqft', 'BHK', 'Bathrooms', 'Latitude', 'Longitude']
print(f"Final Categorical Features: {categorical_features}")
print(f"Final Numerical Features: {numerical_features}")

## 4, 5, & 6. Handle Outliers and Missing Values
We enforce strict domain limits (prices > 0, areas > 0, bhk > 0).

In [ ]:
initial_rows = len(df)

# Domain-based outlier/invalid value removal
df = df[(df['Price_INR'] > 0) & (df['Area_sqft'] > 0)]
df = df[(df['BHK'] > 0) & (df['Bathrooms'] > 0)]

rows_removed = initial_rows - len(df)
print(f"Rows before outlier handling: {initial_rows}")
print(f"Rows removed (domain checks): {rows_removed}")
print(f"Rows remaining: {len(df)}")

## 7, 10 & 11. Categorical Encoding & Pipeline Building
We use **One-Hot Encoding** for categorical variables (`Location` and `Property_Type`).

To ensure reproducibility for deployment (Streamlit), we create a scikit-learn `ColumnTransformer` and save it to the `model/` folder. This guarantees user inputs will be encoded exactly as our training data.

In [ ]:
# 1. Create a deployable Pipeline preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)

X_raw = df[categorical_features + numerical_features]
preprocessor.fit(X_raw)

os.makedirs('../model', exist_ok=True)
with open('../model/preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)
print("Saved preprocessor to: model/preprocessor.pkl")

# 2. Create the explicit ML-ready dataset CSV for direct model training/inspection
df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True)

os.makedirs('../data/processed', exist_ok=True)
output_path = '../data/processed/ml_ready_data.csv'
df_encoded.to_csv(output_path, index=False)
print(f"Saved ml-ready dataset to: {output_path}")

## 8 & 9. Train-Test Split and Data Leakage Prevention
### Data Leakage Prevention
* **Target Isolation:** The target (`Price_INR`) is stripped from the feature matrix `X` before any splitting or training begins.
* **Feature Leakage:** `price_per_sqft` is entirely removed because retaining it provides future knowledge to the model.
* **Separation:** Data is split into 80% Training and 20% Testing. Models will *only* be fitted on the Training data, and Testing data remains completely unseen during training.

In [ ]:
X = df_encoded.drop(columns=[target])
y = df_encoded[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

## 12. Final Validation

In [ ]:
print("--- Final Validation ---")
print(f"Final feature count: {X.shape[1]}")
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")
print(f"Target variable: {target}")
print(f"Numerical features: {numerical_features}")
print(f"Categorical features: {categorical_features} -> (One-Hot Encoded)")
print(f"Missing values in X_train: {X_train.isnull().sum().sum()}")

## 13. Feature Engineering Summary

1. **Selected Features:** `Location`, `Property_Type`, `Area_sqft`, `BHK`, `Bathrooms`, `Latitude`, and `Longitude`. These capture all critical geospatial, physical, and infrastructural attributes of the property.
2. **Removed Features:** `price_per_sqft`.
3. **Categorical Encoding Requirement:** ML models require purely numeric inputs. `Location` and `Property_Type` are categorical text arrays that must be expanded via **One-Hot Encoding** (creating binary 0/1 columns) so regression algorithms can process them.
4. **Numerical Feature Handling:** They were preserved as native floats/ints. Scaling was skipped as tree-based models (often best for real estate) do not require it, and leaving them unscaled simplifies interpretability. 
5. **Outliers Handling:** Negative/zero domain constraints were explicitly enforced to prevent illogical inputs.
6. **Target Leakage Prevention:** `price_per_sqft` was permanently dropped before creating `X`. Keeping it would give the model direct mathematical access to the answer, rendering it useless for real-world predictions.
7. **Train/Test Splitting:** A standard `80/20` split was achieved via Scikit-Learn using a fixed `random_state=42` to guarantee reproducibility across multiple runs.
8. **Regression Suitability:** The dataset now contains zero missing values, zero extreme unhandled outliers, purely numeric features, and strict train-test bounds, meaning it is mathematically perfectly suited for regressors like Random Forest or Linear Regression.